# FidelProbe

Do small VLMs understand the consonant × vowel structure of the Ethiopic script? Qwen3-VL-2B-Instruct on 2,592 probes, with Korean Hangul as a control.

## Setup

In [ ]:
!pip -q install -U "transformers>=4.57" accelerate "huggingface_hub>=1.6"
!apt-get -qq install -y fonts-noto-core fonts-noto-cjk fonts-sil-abyssinica > /dev/null
!git clone -q https://github.com/aleale2121/fatima_challenge.git
%cd fatima_challenge/fidelprobe
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

## Build the probe set

In [ ]:
!python build_dataset.py --out ../data --seed 0

## Run the model

In [ ]:
MODEL = "Qwen/Qwen3-VL-2B-Instruct"
!python run_eval.py --model $MODEL --data ../data --out ../results

## Tokenizer check and scoring

In [ ]:
!python tokenizer_probe.py --model $MODEL --out ../results > /dev/null
!python analyze.py --data ../data --results ../results > /dev/null
from IPython.display import Image, Markdown, display
import glob
display(Markdown(open(glob.glob('../results/RESULTS_*.md')[0]).read().split('![')[0]))
for p in sorted(glob.glob('../results/fig_*.png')): display(Image(p))

## Some Ethiopic mistakes

In [ ]:
import pandas as pd
df = pd.read_json(glob.glob('../results/scored_*.jsonl')[0], lines=True)
df[(df.script == 'eth') & (~df.correct)][['task', 'gold', 'answer', 'raw']].sample(20, random_state=0)

## Upload to the Hugging Face bucket

In [ ]:
!python upload_to_hf.py --bucket fidelprobe --model $MODEL